# 🧠 Building a Probabilistic Digital Twin: Bayesian PI-NODE-SR
Welcome to this interactive guide on upgrading the **Physics-Informed Neural ODE with Scale-Aware Residuals (PI-NODE-SR)** framework to a fully **Bayesian Probabilistic Programming** architecture.

In this notebook, we will explore the mathematical foundations of learning stiff biophysical dynamics (like the Hodgkin-Huxley equations) not by searching for a single "best" neural network, but by mapping a landscape of all physically plausible realities.

---

## 1. The Core Problem: Stiff Biophysical Dynamics
Biological systems are notoriously difficult for standard Neural ODEs to learn because they are **stiff**. Stiffness arises when different variables in a system evolve on vastly disparate timescales[cite: 1].

### The Hodgkin-Huxley (HH) Example
Consider the HH model, which governs the nonlinear dynamics of a neuronal membrane:
*   **Fast Variable:** The membrane voltage ($V$) spikes incredibly quickly in a matter of milliseconds.
*   **Slow Variables:** The ion channel gating recovery variables ($n, m, h$) evolve much more slowly and within a strictly bounded domain (0 to 1)[cite: 1].

When standard machine learning models try to fit this data, they suffer from **numerical diffusion and gradient pathologies**. They tend to overfit to the massive spikes of the voltage variable while completely ignoring the slow, tiny gradients of the gating variables. As a result, standard Neural ODEs fail to generalize, drifting in phase and eventually collapsing to non-spiking subthreshold states[cite: 1].

## 2. The Deterministic Solution: PI-NODE-SR
To fix this gradient imbalance, the PI-NODE-SR framework introduces two critical innovations:

1.  **Scale-Aware Residuals:** Instead of treating all physics equations equally, the physics loss is normalized by a scaling factor $s_{j}$ (traditionally the empirical standard deviation of the derivative)[cite: 1].
2.  **Solver-Loss Synergy:** By balancing the residuals, the model can surprisingly utilize a low-order explicit solver like **Heun's method**, which is typically highly unstable for stiff systems, to accurately extrapolate long-horizon dynamics[cite: 1].

### The Deterministic Loss Function
The deterministic model minimizes a composite loss function:
$$ \mathcal{L}_{total} = \mathcal{L}_{data} + \lambda \mathcal{L}_{physics} $$

Where the physics loss is defined as:
$$ \mathcal{L}_{physics} = \frac{1}{N} \sum_{i=1}^{N} \sum_{j=1}^{4} \left( \frac{r_{ij}}{s_{j}} \right)^{2} $$
*Here, $r_{ij}$ is the residual error of the physics equation, $s_{j}$ is the fixed scaling factor, and $\lambda$ is a fixed hyperparameter balancing the two objectives[cite: 1].*

## 3. Why Upgrade to a Bayesian Framework?
While the deterministic PI-NODE-SR achieves stable long-term extrapolation[cite: 1], it suffers from significant limitations that a Bayesian framework elegantly solves:

*   **Initialization Sensitivity:** The model is highly sensitive to how the neural network weights are initialized, occasionally getting stuck in local minima where the neuron fails to spike[cite: 1].
*   **Manual Tuning:** The scale factors ($s_{j}$) and the physics weight ($\lambda$) must be manually calculated and tuned by the researcher.
*   **No Uncertainty Quantification (UQ):** The model predicts a single trajectory. If the model starts to drift after 100 milliseconds, it gives the researcher no mathematical warning.

### The Bayesian Paradigm Shift
Instead of using optimizers like Adam to find a single point-estimate of the network weights ($\theta$), we will use **Markov Chain Monte Carlo (MCMC)** to find the **Posterior Distribution** of the weights, scales, and hyperparameters. We shift from answering *"What is the equation?"* to answering *"Given the data and the physics, what is the probability distribution of all possible equations?"*

## 4. Defining the Generative Model (The Priors)
In a probabilistic programming framework (like Turing.jl or Pyro), we first define our **Priors**—our mathematical beliefs about the system before we observe any data.

By turning manual hyperparameters into latent random variables, we allow the system to self-regulate.

1.  **The Neural Network Weights ($\theta$):**
    We assume the weights are normally distributed around zero. This acts as equivalent to L2 regularization in classical ML.
    $$ \theta \sim \mathcal{N}(0, \alpha I) $$
2.  **The Scale-Aware Parameters ($s_{j}$):**
    Instead of calculating $s_j$ manually, we assign it a **Half-Cauchy** prior. The Cauchy distribution has heavy tails, which allows the model to dynamically "stretch" the scale if it discovers that a specific variable (like the $m$-gate) is excessively stiff.
    $$ s_{j} \sim \text{Half-Cauchy}(\gamma) \quad \text{for } j \in \{V, n, m, h\} $$
3.  **The Physics Weight ($\lambda$):**
    We treat the balance between data and physics as a random variable governed by a Gamma distribution, ensuring it stays strictly positive.
    $$ \lambda \sim \text{Gamma}(a, b) $$
4.  **Observation Noise ($\sigma_{obs}$):**
    We estimate the inherent biological "jitter" in the data.
    $$ \sigma_{obs} \sim \text{Normal}_{+}(0, \sigma) $$

## 5. The Two-Headed Likelihood Function
To train a Bayesian model, we convert classical loss functions (Mean Squared Error) into **Log-Likelihoods**. We establish two "observation" pathways: one for the empirical data, and one for the physical laws.

### A. The Data Likelihood (Measurement Model)
We state that the noisy biological observations $\tilde{z}(t)$ follow a Normal distribution centered perfectly on the trajectory predicted by our Neural ODE:

$$ p(\mathcal{D} \mid \theta, \sigma_{obs}) = \prod_{i=1}^{N} \mathcal{N} \left( \tilde{z}_{i} \mid \text{ODESolve}(f_{\theta}, z_{0}, t_{i}), \sigma_{obs}^{2} \right) $$

### B. The Physics Likelihood (Virtual Observation)
This is the probabilistic upgrade of the scale-aware residual constraint. We treat the physical Hodgkin-Huxley equations as **virtual observations**. We tell the model: *"We observe that the physics residual $r_{ij}$ must be exactly 0, but we allow a variance defined by our learned scales $s_{j}$."*

$$ p(f_{phys} \mid \theta, s_{j}, \lambda) = \prod_{i=1}^{N} \prod_{j=1}^{4} \mathcal{N} \left( r_{ij} \mid 0, \frac{s_{j}^{2}}{\lambda} \right) $$

> **💡 The Magic of Automatic Relevance Determination (ARD):** 
> If the physics equation for the voltage $V$ is highly mismatched, the residual $r_{iV}$ will be large. To maximize the log-likelihood without breaking the data fit, the Bayesian sampler will automatically push the latent variable $s_{V}$ higher, widening the distribution and dynamically reducing the penalty for that specific stiffness!

## 6. Inference and the Adjoint Method
To explore this highly complex, high-dimensional probability space, we use the **No-U-Turn Sampler (NUTS)**, a highly efficient gradient-based MCMC algorithm.

### The Stiff Gradient Problem
Because NUTS relies on gradients to navigate the probability landscape, we must compute the gradient of the Neural ODE output with respect to its parameters. 

The choice of gradient calculation is critical. As noted in the PI-NODE-SR framework, using standard "backsolve" adjoint methods on stiff spike dynamics results in exploding or vanishing sensitivities, causing the training to collapse entirely[cite: 1]. 

To survive the mathematical violence of a neuronal spike, the Bayesian inference engine must be paired with an **Interpolating Adjoint**. This provides a smooth, spline-based trajectory for backpropagation, allowing the sampler to maintain stable supervision through sharp, stiff transients[cite: 1].

## 7. The Final Product: Uncertainty-Aware Physics
Once the NUTS sampler finishes running, we are left with a posterior distribution of thousands of highly probable parameter sets.

What do we gain from this?

1.  **Confidence Ribbons:** Instead of projecting a single deterministic voltage trace into the future, we draw 100 trajectories from our posterior. As we extrapolate beyond the training window (e.g., past 100ms), the lines will begin to fan out. This visual ribbon tells us exactly *when* the model's physical confidence breaks down.
2.  **Physics Discovery:** By plotting the posterior distributions of the $s_{j}$ scale variables, we can perform structural diagnostics. If $s_m$ has a vastly wider distribution than $s_n$, the model is mathematically proving to us that the sodium activation gate is the primary source of biological stiffness.
3.  **Robustness:** Because MCMC explores the entire loss landscape rather than greedily falling into the nearest local minimum, it entirely circumvents the initialization sensitivity that plagues deterministic PI-NODEs[cite: 1].

*We have successfully upgraded a curve-fitter into a Probabilistic Digital Twin that reasons about its own uncertainty.*

In [ ]:
          # %%
# import Pkg; Pkg.add(["Turing", "Distributions", "LinearAlgebra", "Lux", "DifferentialEquations", "SciMLSensitivity", "Statistics", "DataFrames", "Plots", "Random", "ComponentArrays"])

using Turing, Distributions, LinearAlgebra
using Lux, SciMLSensitivity, ComponentArrays, StaticArrays, Zygote, DifferentialEquations
using Statistics, Random, Plots

In [ ]:
using CSV
using DataFrames

# 1. Define the file path
file_path = "e:/Neural_Spiking_Dynamics/notebooks/1_data_generation/single_spike_noisy_data.csv"

# 2. Read the CSV directly into a DataFrame (Raw Order: timestamp, V, m, n, h)
HH_data_raw = CSV.read(file_path, DataFrame)

# 3. Select all rows (:) and reorder the columns as desired
df_ordered = HH_data_raw[:, [:timestamp, :V, :n, :m, :h]]

# Display the first few rows to verify the new order
first(df_ordered, 5)
          

In [ ]:
# %%
df_ordered = HH_data[:, [:timestamp, :V, :n, :m, :h]]

# %%
t_train = Float32.(df_ordered.timestamp)
z_train = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]])')

In [ ]:

# %%
nn = Lux.Chain(
    Lux.Dense(4 => 16, Lux.tanh), # 4 inputs, NO TIME
    Lux.Dense(16 => 16, Lux.tanh),
    Lux.Dense(16 => 4)            # 4 outputs
)

rng = Random.default_rng()
ps, st = Lux.setup(rng, nn) 

# Flatten parameters for SciML and Turing
p_initial = ComponentArray(ps)
axes_p = getaxes(p_initial) # Save the axes so we can rebuild the ComponentArray inside Turing

In [ ]:
# %%
# Float32 Rate Functions
α_m(V) = 0.1f0 .* (V .+ 40.0f0) ./ (1.0f0 .- exp.(-(V .+ 40.0f0) ./ 10.0f0))
β_m(V) = 4.0f0 .* exp.(-(V .+ 65.0f0) ./ 18.0f0)
α_h(V) = 0.07f0 .* exp.(-(V .+ 65.0f0) ./ 20.0f0)
β_h(V) = 1.0f0 ./ (1.0f0 .+ exp.(-(V .+ 35.0f0) ./ 10.0f0))
α_n(V) = 0.01f0 .* (V .+ 55.0f0) ./ (1.0f0 .- exp.(-(V .+ 55.0f0) ./ 10.0f0))
β_n(V) = 0.125f0 .* exp.(-(V .+ 65.0f0) ./ 80.0f0)

function hh_equations(u)
    V = u[1:1, :] 
    n = u[2:2, :]
    m = u[3:3, :]
    h = u[4:4, :]
    
    I_ext = 10.0f0
    g_Na  = 120.0f0
    g_K   = 36.0f0
    g_L   = 0.3f0
    E_Na  = 50.0f0
    E_K   = -77.0f0
    E_L   = -54.4f0
    C_m   = 1.0f0

    I_Na = g_Na .* (m.^3) .* h .* (V .- E_Na)
    I_K  = g_K .* (n.^4) .* (V .- E_K)
    I_L  = g_L .* (V .- E_L)
    
    dV = (I_ext .- I_Na .- I_K .- I_L) ./ C_m
    dn = α_n.(V) .* (1.0f0 .- n) .- β_n.(V) .* n
    dm = α_m.(V) .* (1.0f0 .- m) .- β_m.(V) .* m
    dh = α_h.(V) .* (1.0f0 .- h) .- β_h.(V) .* h
    
    return vcat(dV, dn, dm, dh)
end

In [ ]:
# %%
function neural_dynamics(u, p, t)
    input_2d = reshape(u, :, 1)
    dudt_2d, _ = nn(input_2d, p, st) 
    return vec(dudt_2d)
end

u0_cpu = Float32[-65.0f0, 0.05f0, 0.6f0, 0.32f0]
tspan = (0.0f0, 50.0f0)

node_prob = ODEProblem(neural_dynamics, u0_cpu, tspan, p_initial)

In [ ]:
# %%
@model function bayesian_pinode(t_train, z_train, prob, st, axes_p)
    # --- 1. Priors (The Latent Variables) ---
    
    # Neural ODE Weights: Normal distribution acting as L2 regularization
    theta ~ MvNormal(Zeros(length(prob.p)), 1.0 * I)
    
    # Scale-Aware Parameters: Half-Cauchy distributions for V, n, m, h
    # This automatically discovers if a variable is stiff and needs scaling
    scales ~ filldist(truncated(Cauchy(0.0, 1.0), lower=0.01), 4)
    
    # Physics Weight: Automatically balances data vs physics
    lambda_phys ~ Gamma(2.0, 0.5)
    
    # Biological Noise / Measurement Error
    sigma_obs ~ truncated(Normal(0.0, 0.5), lower=0.01)

    # --- 2. ODE Solve ---
    # Reconstruct the Lux parameters from the flat theta vector
    p_sampled = ComponentArray(theta, axes_p)
    new_prob = remake(prob, p=p_sampled)
    
    # Using Heun with InterpolatingAdjoint to handle stiff spikes without collapsing[cite: 1]
    sol = solve(new_prob, Heun(), saveat=t_train, reltol=1e-5, abstol=1e-5, sensealg=InterpolatingAdjoint())
    
    # If the solver fails (diverges), explicitly reject these parameters
    if sol.retcode != ReturnCode.Success
        Turing.@addlogprob! -Inf
        return
    end
    
    pred_z = Array(sol)
    
    # --- 3. Data Likelihood (Measurement Model) ---
    for i in 1:size(z_train, 2)
        for j in 1:4
            z_train[j, i] ~ Normal(pred_z[j, i], sigma_obs)
        end
    end

    # --- 4. Scale-Aware Physics Likelihood (Virtual Observation) ---
    # Forward pass on the predictions to get the learned vector field
    pred_derivs, _ = nn(pred_z, p_sampled, st)
    
    # Calculate the ground truth physics on those predictions
    true_derivs = hh_equations(pred_z)
    
    # Calculate Residuals
    residuals = pred_derivs .- true_derivs
    
    # We "observe" that the residual MUST be 0. 
    # The variance is dynamically managed by our sampled scales and lambda.
    for i in 1:size(residuals, 2)
        for j in 1:4
            0.0 ~ Normal(residuals[j, i], scales[j] / lambda_phys)
        end
    end
end

In [ ]:
# %%
# Instantiate the model
prob_model = bayesian_pinode(t_train, z_train, node_prob, st, axes_p)

println("Starting NUTS MCMC Sampling...")
# We use NUTS (No-U-Turn Sampler), which utilizes Zygote/SciML gradients
# 500 warmup steps to find the typical set, then draw 500 valid samples
chain = sample(prob_model, NUTS(0.65), 1000)

println("Bayesian Inference Complete!")

In [ ]:
# %%
# Plot the learned latent variables
# You will see the model "discovered" different scales for V vs n, m, h
plot(chain[["scales[1]", "scales[2]", "scales[3]", "scales[4]"]], 
     title="Posterior Distribution of Scale Factors (s_j)")

plot(chain[["lambda_phys", "sigma_obs"]], 
     title="Posterior of Physics Weight & Data Noise")